## Setup and Imports

In [28]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,
    EPOCHS_LSTM   = 4,
    BS_LSTM       = 256,
    MAX_LEN_DEB   = 512,
    LR_LSTM       = 1e-3,
    LSTM_HIDDEN   = 384,
    MARGIN        = 0.5,
    LSTM_EMBED    = 256,
    USE_BF16      = True,
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
    ARTIFACT_DIR  = '/kaggle/working/artifacts'
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)
OPTS = ['A', 'B', 'C', 'D', 'E']

In [29]:
import os, sys, json, math, random, time, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler

from sklearn.metrics import f1_score, accuracy_score


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU: {p.name}, {p.total_memory/1e9:.1f} GB, compute {p.major}.{p.minor}")

if CFG['USE_BF16'] and DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported():
    AMP_DTYPE = torch.bfloat16
elif DEVICE.type == 'cuda':
    AMP_DTYPE = torch.float16
else:
    AMP_DTYPE = torch.float32
USE_SCALER = (AMP_DTYPE == torch.float16)
print(f"AMP dtype: {AMP_DTYPE} | GradScaler: {USE_SCALER}")

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch.set_float32_matmul_precision('high')

def autocast():
    return _autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda'))

def make_scaler():
    return _GradScaler('cuda', enabled=USE_SCALER)

Device: cuda
  GPU: Tesla T4, 15.6 GB, compute 7.5
AMP dtype: torch.bfloat16 | GradScaler: False


In [30]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
y_idx = np.array([OPTS.index(a) for a in train['answer']])

In [31]:
import re
from sklearn.model_selection import GroupKFold

_BOILERPLATE = [
    r"pick the best possible answer:?",
    r"\bcarefully\b\.?",
    r"among the listed options\.?",
    r"choose the (correct|best) (option|answer|response)\.?",
    r"select the (best|correct|most appropriate) (option|answer|response)\.?",
    r"determine the correct option:?",
    r"identify the correct statement:?",
    r"based on the given context\.?",
    r"which of the following( statements)?( is true)?",
]
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = str(text)
    for pat in _BOILERPLATE:
        s = re.sub(pat, ' ', s, flags=re.IGNORECASE)
    return _WS.sub(' ', s).strip()

for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)


gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.loc[train.index[va_idx], 'fold'] = fold_id

assert (train['fold'] >= 0).all()
_dup = train.groupby('prompt_n')['fold'].nunique()
assert _dup.max() == 1, "Leak: a prompt spans multiple folds"
print("Folds:\n", train['fold'].value_counts().sort_index().to_string())

Folds:
 fold
0    1000
1    1000


In [32]:
def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = list(order[i, :3])
        if y_idx[i] in top3:
            total += 1.0 / (top3.index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    pred = scores_2d.argmax(axis=1)
    return dict(map3=map_at_3(scores_2d, y_idx),
                acc=accuracy_score(y_idx, pred),
                f1=f1_score(y_idx, pred, average='macro'))

In [33]:
def pairwise_margin_loss(scores, correct, margin=0.5):
    """Push the correct option's score above every distractor by >= margin."""
    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss)
    mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)

### Tracking (Weights & Biases)


In [34]:
import os
try:
    import wandb
except ImportError:
    wandb = None

USE_WANDB = True
RUN_VERSION = 1
RUN_NOTES = "baseline run , n_folds = 2, epochs = 4"         
WANDB_PROJECT = "smart-mcq"

if USE_WANDB and wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("wandb_api"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}); set Kaggle secret 'wandb_api' to enable.")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",    
        job_type="train",
        notes=RUN_NOTES,                               
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION, "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B connected.


### Uploading Model to Kagglehub

In [35]:
try:
    import kagglehub
except ImportError:
    kagglehub = None


KAGGLE_USERNAME = "kamal134134"
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    "lstm":    f"{KAGGLE_USERNAME}/smart-mcq-lstm/{HUB_FRAMEWORK}/baseline"
}

def push_model(model_tag, folder, notes=""):
    if not PUSH_TO_HUB or kagglehub is None:
        print(f"[hub] upload skipped for '{model_tag}' (PUSH_TO_HUB=False).")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"[hub] uploading '{model_tag}' -> {handle}")
    kagglehub.model_upload(handle, folder, version_notes=note)
    print(f"[hub] done. Inference notebooks can now kagglehub.model_download('{handle}').")

## 1. Vocabulary

In [36]:
PAD, UNK = 0, 1
LSTM_MAX_OPT_LEN = 25
LSTM_MAX_Q_LEN   = 80

def build_vocab(texts, min_freq=2, max_size=30000):
    cnt = Counter()
    for t in texts:
        cnt.update(re.findall(r"[a-zA-Z']+|\d+", t.lower()))
    vocab = {'<pad>': PAD, '<unk>': UNK}
    for w, c in cnt.most_common(max_size):
        if c >= min_freq:
            vocab[w] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len):
    toks = re.findall(r"[a-zA-Z']+|\d+", text.lower())[:max_len]
    ids = [vocab.get(t, UNK) for t in toks]
    ids += [PAD] * (max_len - len(ids))
    return np.array(ids, dtype=np.int64)

_all = [train['prompt_n'].tolist(), test['prompt_n'].tolist()]
for o in OPTS:
    _all += [train[f'{o}_n'].tolist(), test[f'{o}_n'].tolist()]
VOCAB = build_vocab([t for grp in _all for t in grp])
print(f"LSTM vocab size: {len(VOCAB)}")

LSTM vocab size: 2983


## 2. Dataset 

In [37]:
class LSTMDataset(Dataset):
    def __init__(self, df, vocab, shuffle_opts=True, has_label=True, seed=0):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.shuffle = shuffle_opts
        self.has_label = has_label
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        q_ids = encode_text(r['prompt_n'], self.vocab, LSTM_MAX_Q_LEN)
        opt_ids = np.stack([encode_text(r[f'{o}_n'], self.vocab, LSTM_MAX_OPT_LEN)
                            for o in OPTS])
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            opt_ids = opt_ids[perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])
        return dict(q=torch.from_numpy(q_ids),
                    opts=torch.from_numpy(opt_ids),
                    correct=torch.tensor(correct, dtype=torch.long),
                    id=int(r['id']))

In [38]:
class SiameseBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden=384, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True,
                            bidirectional=True, num_layers=1)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(4 * 2 * hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, 1),
        )

    def encode(self, ids):
        emb = self.embed(ids)
        out, _ = self.lstm(emb)
        mask = (ids != PAD).unsqueeze(-1).float()
        out = out * mask + (1 - mask) * -1e9
        pooled, _ = out.max(dim=1)
        return self.dropout(pooled)

    def forward(self, q_ids, opt_ids):
        B = q_ids.size(0)
        h_q = self.encode(q_ids)
        h_o = self.encode(opt_ids.view(B * 5, -1)).view(B, 5, -1)
        h_q = h_q.unsqueeze(1).expand(-1, 5, -1)
        joined = torch.cat([h_q, h_o, torch.abs(h_q - h_o), h_q * h_o], dim=-1)
        return self.head(joined).squeeze(-1)

## 4. Cross-validated training

Pairwise margin loss, best-epoch checkpointing per fold, OOF + test predictions accumulated and saved to disk.

In [39]:
MODEL_TAG  = 'lstm'
UPLOAD_DIR = f"{CFG['ARTIFACT_DIR']}/{MODEL_TAG}"
os.makedirs(UPLOAD_DIR, exist_ok=True)


@torch.no_grad()
def predict_lstm(model, dl):
    model.eval(); out = []
    for batch in dl:
        q = batch['q'].to(DEVICE); opts = batch['opts'].to(DEVICE)
        with autocast():
            s = model(q, opts)
        out.append(s.cpu().float().numpy())
    return np.concatenate(out)


def train_lstm_cv(train_df, test_df, n_folds=CFG['N_FOLDS'], save_dir=UPLOAD_DIR):
    oof = np.zeros((len(train_df), 5), dtype=np.float32)
    te_pred = np.zeros((len(test_df), 5), dtype=np.float32)
    y = np.array([OPTS.index(a) for a in train_df['answer']])
    te_dl = DataLoader(LSTMDataset(test_df, VOCAB, shuffle_opts=False, has_label=False),
                       batch_size=CFG['BS_LSTM'], shuffle=False)

    for fold in range(n_folds):
        run = wandb_init(MODEL_TAG, fold)       
        va_idx = np.where(train_df['fold'].values == fold)[0]
        tr_idx = np.where(train_df['fold'].values != fold)[0]
        tr_dl = DataLoader(LSTMDataset(train_df.iloc[tr_idx], VOCAB, True, seed=fold),
                           batch_size=CFG['BS_LSTM'], shuffle=True, pin_memory=True)
        va_dl = DataLoader(LSTMDataset(train_df.iloc[va_idx], VOCAB, False),
                           batch_size=CFG['BS_LSTM'], shuffle=False)

        model = SiameseBiLSTM(len(VOCAB), CFG['LSTM_EMBED'], CFG['LSTM_HIDDEN']).to(DEVICE)
        optim = AdamW(model.parameters(), lr=CFG['LR_LSTM'], weight_decay=1e-5)
        sched = CosineAnnealingLR(optim, T_max=CFG['EPOCHS_LSTM'] * max(len(tr_dl), 1))
        scaler = make_scaler()

        best_map3, best_va, best_te = -1, None, None
        for epoch in range(CFG['EPOCHS_LSTM']):
            model.train(); running, seen = 0.0, 0
            for batch in tr_dl:
                q = batch['q'].to(DEVICE); opts = batch['opts'].to(DEVICE)
                correct = batch['correct'].to(DEVICE)
                optim.zero_grad()
                with autocast():
                    loss = pairwise_margin_loss(model(q, opts), correct, CFG['MARGIN'])
                scaler.scale(loss).backward()
                if USE_SCALER: scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optim); scaler.update(); sched.step()
                running += loss.item() * q.size(0); seen += q.size(0)

            va_scores = predict_lstm(model, va_dl)
            m = all_metrics(va_scores, y[va_idx])
            wlog({'train_loss': running / max(seen, 1), 'val_map3': m['map3'],
                  'val_acc': m['acc'], 'val_f1': m['f1'], 'epoch': epoch}, step=epoch)
            print(f"  fold {fold} ep {epoch}: MAP@3={m['map3']:.4f} acc={m['acc']:.4f}")
            if m['map3'] > best_map3:
                best_map3, best_va = m['map3'], va_scores
                best_te = predict_lstm(model, te_dl)
                torch.save(model.state_dict(), f"{save_dir}/best_lstm_fold{fold}.pth")

        wlog({'best_val_map3': best_map3})
        oof[va_idx] = best_va; te_pred += best_te / n_folds
        wandb_finish()
        del model; gc.collect(); torch.cuda.empty_cache()
    return oof, te_pred


oof_lstm, test_lstm = train_lstm_cv(train, test)
print(f"\nLSTM OOF MAP@3: {map_at_3(oof_lstm, y_idx):.4f}")
np.save(f"{UPLOAD_DIR}/oof_lstm.npy",  oof_lstm)
np.save(f"{UPLOAD_DIR}/test_lstm.npy", test_lstm)
push_model(MODEL_TAG, UPLOAD_DIR,
           notes=f"LSTM v{RUN_VERSION} | OOF MAP@3={map_at_3(oof_lstm, y_idx):.4f} | {RUN_NOTES}")

  fold 0 ep 0: MAP@3=0.7258 acc=0.5980
  fold 0 ep 1: MAP@3=0.8045 acc=0.7160
  fold 0 ep 2: MAP@3=0.8457 acc=0.7750
  fold 0 ep 3: MAP@3=0.8512 acc=0.7820


best_val_map3,▁
epoch,▁▃▆█
train_loss,█▅▂▁
val_acc,▁▅██
val_f1,▁▅██
val_map3,▁▅██
best_val_map3,0.85117
epoch,3
train_loss,0.17834
val_acc,0.782
val_f1,0.78056


  fold 1 ep 0: MAP@3=0.7742 acc=0.6540
  fold 1 ep 1: MAP@3=0.8097 acc=0.6980
  fold 1 ep 2: MAP@3=0.8260 acc=0.7310
  fold 1 ep 3: MAP@3=0.8282 acc=0.7350


best_val_map3,▁
epoch,▁▃▆█
train_loss,█▄▂▁
val_acc,▁▅██
val_f1,▁▅██
val_map3,▁▆██
best_val_map3,0.82817
epoch,3
train_loss,0.15323
val_acc,0.735
val_f1,0.73216



LSTM OOF MAP@3: 0.8397
[hub] uploading 'lstm' -> kamal134134/smart-mcq-lstm/pyTorch/baseline
Uploading Model https://api.kaggle.com/models/kamal134134/smart-mcq-lstm/pyTorch/baseline ...
Model 'smart-mcq-lstm' does not exist or access is forbidden for user 'kamal134134'. Creating or handling Model...
Starting upload for file /kaggle/working/artifacts/lstm/best_lstm_fold1.pth


...


Uploading: 100%|██████████| 40.1k/40.1k [00:00<00:00, 202kB/s]


Upload successful: /kaggle/working/artifacts/lstm/oof_lstm.npy (39KB)
Starting upload for file /kaggle/working/artifacts/lstm/test_lstm.npy


Uploading: 100%|██████████| 10.1k/10.1k [00:00<00:00, 57.2kB/s]

Upload successful: /kaggle/working/artifacts/lstm/test_lstm.npy (10KB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/kamal134134/smart-mcq-lstm/pyTorch/baseline
[hub] done. Inference notebooks can now kagglehub.model_download('kamal134134/smart-mcq-lstm/pyTorch/baseline').
